# Create NEH Awards (GRANT PATTERN, bulk-CSV method-4)

The National Endowment for the Humanities is the largest US federal
funder of humanities scholarship (~$170M/yr, established 1965). NEH
publishes their complete grant history (1965-present) as bulk CSV on
an Azure Blob storage they own and operate themselves —
`https://nehopendatastorage.blob.core.windows.net/nehopendata/NEH_Grants{decade}.csv`,
linked from `https://securegrants.neh.gov/open/data/`. This is **NEH's
own publishing channel, not USAspending or any other federal
aggregator** — i.e., it satisfies the direct-source rule unlike the
USAspending-suffixed federal funders at priorities 53-60.

Fills the humanities gap in a registry that has been heavily science-
biased (NIH, NSF, DOE, NIST, NSERC, ARC, ERC, etc.). The closest
existing humanities-leaning entries are SSHRC (#5) and Lasker (#48,
biomedical) — none directly cover US humanities.

**Awarding body:** National Endowment for the Humanities — F4320306100 (US)

**Schema choices:**
- One row per NEH grant (the `AppNumber` is the canonical funder award
  identifier). Slug-collision raises in the source script if any
  duplicates appear.
- `funder_scheme` = NEH `Program` (e.g., `'Fellowships for Younger
  Scholars'`, `'Infrastructure and Capacity Building Challenge
  Grants'`). NEH organizes its funding into roughly two dozen named
  programs across six divisions.
- `amount` = `AwardOutright + AwardMatching + SupplementAmount` —
  the actual disbursed amount (sums to NEH's published yearly totals).
  `OriginalAmount` is preserved in the staging table for traceability
  but not used as the primary amount.
- `currency = 'USD'` hardcoded (NEH is US-only).
- `start_date` / `end_date` parsed from `BeginGrant` / `EndGrant`
  (NEH date format `M/D/YYYY HH:MM:SS AM/PM`, normalized to ISO in the
  source script).
- `lead_investigator` = first Project Director in the `Participants`
  field (`'Name [Role]; Name [Role]; ...'`), with given/family
  parsed via `split_name`.
- `description` = `ToSupport` concatenated with HTML-stripped
  `ProjectDesc` when both populated.
- `declined` boolean populated (always False; NEH doesn't publish
  declined-grant data).

**Source:** https://securegrants.neh.gov/open/data/
**S3 location:** `s3a://openalex-ingest/awards/neh/neh_grants.parquet`
**Method-4 (bulk CSV) on the runbook ladder** — direct, documented
(NEH provides a GrantsDictionary.pdf), and stable. Cleanest possible
federal-funder source.

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.neh_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/neh/neh_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.neh_raw;

## Step 1.5: Inspect raw + money/currency scan

All source columns ship as STRING per runbook §1.2.5. The script
pre-computes a canonical `amount_usd` column (sum of `AwardOutright +
AwardMatching + SupplementAmount`) and a `currency` column ('USD' or
NULL). The §1.5 money-shape scan below sanity-checks the
distribution to catch any future schema drift in NEH's CSVs.

Expected amount distribution: spans 1965-present, USD only, range
roughly $1K-$10M per grant, average in the $50K-$200K bracket.
If min/max look wrong, the source script may have a unit-encoding
bug — re-inspect before transforming.

Source columns preserved for trace: `award_outright`, `award_matching`,
`supplement_amount`, `original_amount` (all STRING). Composite
`amount_usd` is the ship column.

In [ ]:
%sql
DESCRIBE openalex.awards.neh_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.neh_raw LIMIT 5;

In [ ]:
%sql
-- §1.5 money-shape scan — confirm amount_usd is in the expected USD-grant range.
SELECT
    MIN(TRY_CAST(amount_usd AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount_usd AS DOUBLE)) AS max_amount,
    AVG(TRY_CAST(amount_usd AS DOUBLE)) AS avg_amount,
    COUNT(amount_usd) AS non_null,
    COUNT(*) AS total_rows
FROM openalex.awards.neh_raw;

In [ ]:
%sql
-- Currency check — should be only 'USD' or NULL.
SELECT currency, COUNT(*) AS rows FROM openalex.awards.neh_raw GROUP BY currency;

## Step 1.6: Fail-fast — verify NEH funder row exists

In [ ]:
%sql
-- Must return exactly 1 row. If 0, STOP — funder missing from openalex.common.funder.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320306100;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.neh_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306100  -- National Endowment for the Humanities
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    n.project_title AS display_name,
    n.description,
    f.funder_id,
    n.funder_award_id,
    TRY_CAST(n.amount_usd AS DOUBLE) AS amount,
    n.currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    -- NEH funds research (humanities scholarship). Distinguish 'fellowship'
    -- programs from research grants via the program name.
    CASE
        WHEN LOWER(n.program) LIKE '%fellowship%' THEN 'fellowship'
        WHEN LOWER(n.program) LIKE '%training%' OR LOWER(n.program) LIKE '%institute%' THEN 'training'
        ELSE 'research'
    END AS funding_type,
    n.program AS funder_scheme,
    'neh_open_data' AS provenance,
    TRY_TO_DATE(n.begin_grant, 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(n.end_grant, 'yyyy-MM-dd') AS end_date,
    TRY_CAST(n.year_awarded AS INT) AS start_year,
    TRY_CAST(SUBSTRING(n.end_grant, 1, 4) AS INT) AS end_year,
    CASE
        WHEN n.lead_full_name IS NULL OR n.lead_full_name = '' THEN NULL
        ELSE struct(
            n.lead_given_name AS given_name,
            n.lead_family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            TRY_TO_DATE(n.begin_grant, 'yyyy-MM-dd') AS role_start,
            struct(
                n.institution AS name,
                n.inst_country AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.neh_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.project_title IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 81

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'neh_open_data' AND priority = 81;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    81 AS priority  -- NEH priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.neh_awards;

## Step 6: Verification

Full §6.1-6.7 — NEH amount-coverage is NOT waived. Expect:
- Row count ~57000 (1965-present total)
- ~95-99% `pct_amount` (NEH publishes amounts; small fraction of
  cancelled/declined-after-award grants may lack them)
- `currencies = [USD]`
- Year range 1965-current year

In [ ]:
%sql
SELECT COUNT(*) AS total_neh_award_rows FROM openalex.awards.neh_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.neh_awards;

In [ ]:
%sql
-- §6.3 Data completeness (runbook canonical query)
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) AS pct_title,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) AS pct_dates,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) AS pct_description
FROM openalex.awards.neh_awards;

In [ ]:
%sql
-- §6.7 amount + currency fail-fast check (NOT waived).
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    ROUND(AVG(amount), 0) AS avg_amount
FROM openalex.awards.neh_awards;

In [ ]:
%sql
-- Sample top-funded grants for inspection
SELECT id, SUBSTRING(display_name, 1, 70) AS title,
       funder_scheme AS program, funding_type, amount, start_year, end_year,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       SUBSTRING(lead_investigator.affiliation.name, 1, 40) AS institution
FROM openalex.awards.neh_awards
WHERE amount IS NOT NULL
ORDER BY amount DESC
LIMIT 12;

In [ ]:
%sql
-- Program distribution — expect a couple dozen distinct funder_schemes
SELECT funder_scheme AS program, COUNT(*) AS rows,
       ROUND(SUM(amount) / 1e6, 1) AS total_usd_millions
FROM openalex.awards.neh_awards
GROUP BY funder_scheme
ORDER BY rows DESC
LIMIT 20;

In [ ]:
%sql
-- Year distribution by decade — confirms 1965-present coverage
SELECT FLOOR(start_year/10)*10 AS decade, COUNT(*) AS rows,
       ROUND(SUM(amount)/1e6, 0) AS total_usd_millions
FROM openalex.awards.neh_awards
WHERE start_year IS NOT NULL
GROUP BY decade
ORDER BY decade;

In [ ]:
%sql
-- Funder struct populated correctly?
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.neh_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;